<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/vector_stores/DXMTIndexDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DXMT Vector Store

[DXMT](https://github.com/dxmt/dxmt) is an open-source vector database: one binary, a write-ahead log with snapshots, HNSW and HNSW-PQ indexes, BM25 full-text search with hybrid fusion, and metadata filters. This notebook shows how to use it as a LlamaIndex vector store: build an index, query it, filter on metadata, run a hybrid (vector + keyword) query, and delete.

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index llama-index-vector-stores-dxmt

### Start a DXMT server

The vector store talks to a server over HTTP. Download a release binary (Linux) or run the Docker image, and start it on the default port `9999`:

```bash
# Linux binary
v=1.6.0; arch=$(uname -m)
curl -L https://github.com/dxmt/dxmt/releases/download/v$v/dxmt-vectordb-$v-$arch-linux-gnu.tar.gz | tar xz
./dxmt-vectordb-$v-$arch-linux-gnu/vectordb serve

# or Docker (dxmtai/dxmt on Docker Hub; the data lives in the volume)
docker run -d --name dxmt -p 9999:9999 -v dxmt:/data dxmtai/dxmt:latest
```

The host and port the store connects to:

In [ ]:
import os

DXMT_HOST = os.environ.get("DXMT_HOST", "localhost")
DXMT_PORT = int(os.environ.get("DXMT_PORT", "9999"))

### Setup OpenAI
The first step is to configure the OpenAI key. It will be used to create embeddings for the documents loaded into the index.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-..."

Download Data

In [ ]:
!mkdir -p 'data/paul_graham/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'

### Loading documents
Load the documents stored in `data/paul_graham/` using the SimpleDirectoryReader.

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("./data/paul_graham/").load_data()
print("Document ID:", documents[0].doc_id, "Document Hash:", documents[0].hash)

### Create the index

`DXMTVectorStore` takes a `table_name` and, for a table that does not exist yet, the embedding `dimension` (1536 for OpenAI `text-embedding-ada-002`, the LlamaIndex default). `distance_metric` is `cosine` by default. `enable_full_text_search=True` adds a BM25 index over the node text, which the hybrid query below uses.

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.dxmt import DXMTVectorStore

vector_store = DXMTVectorStore(
    table_name="paul_graham",
    dimension=1536,
    host=DXMT_HOST,
    port=DXMT_PORT,
    enable_full_text_search=True,
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

### Query the index
We can now ask questions using our index.

In [ ]:
query_engine = index.as_query_engine()
response = query_engine.query("How much did Viaweb charge per month?")
print(response)

In [ ]:
response = query_engine.query("What did the author do growing up?")
print(response)

### Metadata filters

Add a few nodes with metadata and filter on it. Every LlamaIndex operator is supported with its documented meaning: `ALL` on an array field is *every* value, `ANY` at least one, `CONTAINS` membership, `TEXT_MATCH` a substring, `IS_EMPTY` absent or empty, and `FilterCondition.NOT` "none of these". A filter the server has no flat form for -- `(a AND b) OR c` -- raises `ValueError` rather than being approximated.

In [ ]:
from llama_index.core.schema import TextNode
from llama_index.core.vector_stores.types import (
    FilterCondition,
    FilterOperator,
    MetadataFilter,
    MetadataFilters,
)

nodes = [
    TextNode(
        text="Retrieval augmented generation combines search with a language model.",
        metadata={"tags": ["rag", "llm"], "year": 2023},
    ),
    TextNode(
        text="BM25 ranks documents by term frequency and document length.",
        metadata={"tags": ["search"], "year": 2009},
    ),
    TextNode(
        text="HNSW builds a layered proximity graph for nearest-neighbour search.",
        metadata={"tags": ["search", "ann"], "year": 2016},
    ),
]
index.insert_nodes(nodes)

retriever = index.as_retriever(
    similarity_top_k=5,
    filters=MetadataFilters(
        filters=[
            MetadataFilter(
                key="tags", value=["search"], operator=FilterOperator.ALL
            ),
            MetadataFilter(
                key="year", value=2010, operator=FilterOperator.GTE
            ),
        ],
        condition=FilterCondition.AND,
    ),
)
for hit in retriever.retrieve("nearest neighbour search"):
    print(hit.metadata, hit.text[:60])

### Hybrid search

With `enable_full_text_search=True` the table has a BM25 index beside the vectors, and `VectorStoreQueryMode.HYBRID` fuses the two rankings on the server. `alpha` is the share of the score that comes from the vector side; leave it unset for the server default.

In [ ]:
from llama_index.core.vector_stores.types import VectorStoreQueryMode

hybrid_retriever = index.as_retriever(
    vector_store_query_mode=VectorStoreQueryMode.HYBRID,
    similarity_top_k=3,
)
for hit in hybrid_retriever.retrieve("Viaweb monthly price"):
    print(round(hit.score, 3), hit.text[:80].replace("\n", " "))

### MMR

`VectorStoreQueryMode.MMR` asks the server for a diversified page: `mmr_threshold` is the usual λ (1 is the plain ranking, 0 the most diverse).

In [ ]:
mmr_retriever = index.as_retriever(
    vector_store_query_mode=VectorStoreQueryMode.MMR,
    similarity_top_k=3,
    vector_store_kwargs={"mmr_threshold": 0.5},
)
for hit in mmr_retriever.retrieve("what the author worked on"):
    print(round(hit.score, 3), hit.text[:80].replace("\n", " "))

### Delete

`delete_nodes(filters=...)` removes everything a filter matches in one request; `delete(ref_doc_id)` removes every node of one document; `clear()` drops the table.

In [ ]:
vector_store.delete_nodes(
    filters=MetadataFilters(
        filters=[
            MetadataFilter(
                key="tags", value="search", operator=FilterOperator.CONTAINS
            )
        ]
    )
)
print(
    len(
        vector_store.get_nodes(
            filters=MetadataFilters(
                filters=[
                    MetadataFilter(
                        key="year", value=2000, operator=FilterOperator.GTE
                    )
                ]
            )
        )
    ),
    "tagged nodes left",
)

vector_store.clear()